# Retail Demand Forecasting and Inventory Optimisation

**Decision:** improve replenishment under limited purchasing funds. This notebook reads actual executed analysis outputs. Set `REBUILD=True` to regenerate forecasting and simulations. The separate Spark feature pipeline is documented in the README.

In [1]:
from pathlib import Path
import json, runpy
import pandas as pd
R = Path('results')
REBUILD = False
if REBUILD or not (R / 'diagnostics.json').exists():
    _ = runpy.run_path('run_analysis.py', run_name='__main__')
print((R / 'diagnostics.json').read_text())

{
  "skus": 60,
  "item_days": 114780,
  "store": "CA_1",
  "departments": [
    "FOODS_1",
    "FOODS_2",
    "FOODS_3"
  ],
  "validation_days": 84,
  "test_days": 84,
  "forecast_horizon": 28,
  "test_start": "2016-02-01",
  "test_end": "2016-04-24",
  "selected_model": "Gradient boosting",
  "optimisation_runs": 108,
  "optimal_solver_runs": 108,
  "budget_breaches": 0,
  "engine_audit": {
    "rows": 114780,
    "units_reconciled": 1337156,
    "engine": "PySpark; verified against pandas and SQLite",
    "spark_version": "3.5.7",
    "rolling_features_include_origin_day": "Allowed: origin-day sales are observed when the forecast is made at close of day",
    "price_rule": "Current observed weekly price, forward-filled only; future prices excluded from model inputs",
    "missing_price_after_day_365": 0,
    "sql_rolling_parity": "passed",
    "spark_rolling_parity": "passed"
  }
}


## 1 Data engineering and scope

The 60-product cohort is selected with information ending at day 1605, before all validation and test origins. PySpark joins and rolling windows are cross-checked against pandas and SQLite. The modelling export uses the verified Spark feature values.

In [2]:
print((R / 'feature_audit.json').read_text())
print(pd.read_csv('data/selection_audit.csv').groupby('dept_id').agg(products=('item_id','size'), selection_period_units=('training_units','sum')).to_string())

{
  "rows": 114780,
  "units_reconciled": 1337156,
  "engine": "PySpark; verified against pandas and SQLite",
  "spark_version": "3.5.7",
  "rolling_features_include_origin_day": "Allowed: origin-day sales are observed when the forecast is made at close of day",
  "price_rule": "Current observed weekly price, forward-filled only; future prices excluded from model inputs",
  "missing_price_after_day_365": 0,
  "sql_rolling_parity": "passed",
  "spark_rolling_parity": "passed"
}
         products  selection_period_units
dept_id                                  
FOODS_1        20                   41243
FOODS_2        20                   51076
FOODS_3        20                  169284


## 2 SQL aggregation

The queryable database includes raw sales, prices and calendar tables, a joined daily view, trailing-window features and department totals.

In [3]:
import sqlite3
with sqlite3.connect(R / 'retail.sqlite') as con:
    overview = pd.read_sql_query('SELECT * FROM department_monthly_sales WHERE year = 2016 ORDER BY month, dept_id', con)
print(overview.to_string(index=False))

dept_id  year  month  units  item_days
FOODS_1  2016      1   2323        620
FOODS_2  2016      1   3595        620
FOODS_3  2016      1  10786        620
FOODS_1  2016      2   2377        580
FOODS_2  2016      2   3132        580
FOODS_3  2016      2  10742        580
FOODS_1  2016      3   2770        620
FOODS_2  2016      3   3852        620
FOODS_3  2016      3  11521        620
FOODS_1  2016      4   1572        480
FOODS_2  2016      4   3314        480
FOODS_3  2016      4   9685        480


## 3 Validation and the untouched test sequence

Models are selected on three earlier 28-day windows, then evaluated on three later windows. Refitting at later origins may use sales observed in completed earlier windows. Every training label ends no later than its forecast origin.

In [4]:
print((R / 'locked_selection.json').read_text())
cutoffs = pd.read_csv(R / 'training_cutoff_audit.csv')
assert (cutoffs.latest_training_label <= cutoffs.origin).all()
print(cutoffs.to_string(index=False))

{
  "model": "Gradient boosting",
  "criterion": "Mean WAPE across three validation origins only",
  "validation_origins": [
    1745,
    1773,
    1801
  ],
  "test_origins": [
    1829,
    1857,
    1885
  ],
  "seed": 20260923,
  "training_label_rule": "All training target days <= forecast origin",
  "inventory_policy_assumptions": {
    "base_lead_days": 2,
    "review_days": 7,
    "base_budget_factor": 1.0,
    "purchase_cost_fraction_of_price": 0.6,
    "daily_holding_cost_fraction_of_unit_cost": 0.001,
    "lost_sales_penalty_fraction_of_price": 0.5
  }
}
 origin  first_training_origin  last_training_origin  latest_training_label  training_examples
   1745                    989                  1717                   1745             176400
   1773                   1017                  1745                   1773             176400
   1801                   1045                  1773                   1801             176400
   1829                   1073                  

## 4 Forecast accuracy

The selected model is not changed after seeing test results. All baselines remain visible. The reported simple mean RMSSE is not the official M5 WRMSSE.

![Forecast comparison](results/figures/forecast_comparison.png)

In [5]:
print(pd.read_csv(R / 'forecast_comparison.csv').to_string(index=False))
print('\nPer-origin results:')
print(pd.read_csv(R / 'forecast_metrics.csv').to_string(index=False))

     split                 model     wape      bias  mean_rmsse
      test     Gradient boosting 0.439138  0.044230    0.697865
      test      Weekday mean 56d 0.437204 -0.055137    0.722653
      test Weekly seasonal naive 0.506111 -0.027415    0.878746
validation     Gradient boosting 0.497295  0.113062    0.709063
validation      Weekday mean 56d 0.564789  0.116780    0.779589
validation Weekly seasonal naive 0.596610  0.006226    0.894716

Per-origin results:
 origin      split                 model     wape      bias  mean_rmsse
   1745 validation Weekly seasonal naive 0.557857  0.051784    0.837863
   1745 validation      Weekday mean 56d 0.538838  0.177279    0.778249
   1745 validation     Gradient boosting 0.466542  0.117080    0.690094
   1773 validation Weekly seasonal naive 0.641101  0.071625    1.018535
   1773 validation      Weekday mean 56d 0.632985  0.185929    0.843250
   1773 validation     Gradient boosting 0.543890  0.151788    0.789840
   1801 validation Weekly s

## 5 Prediction intervals

Item-level validation residual quantiles create nominal 80% intervals. Held-out empirical coverage is reported without test-based recalibration.

![Forecast timeline](results/figures/forecast_timeline.png)

In [6]:
print(pd.read_csv(R / 'interval_coverage.csv').to_string(index=False))

 origin  nominal_coverage  empirical_coverage  mean_width
   1829               0.8            0.749405   11.256441
   1857               0.8            0.772619   11.224199
   1885               0.8            0.743452   11.212924


## 6 Inventory decisions

All policies share the same budget rule and starting conditions within each scenario. Operating cost is holding cost plus assumed shortage penalty. Procurement spending and inventory value are separate.

![Inventory comparison](results/figures/inventory_comparison.png)

In [7]:
inventory = pd.read_csv(R / 'inventory_base_case.csv')
print(inventory.to_string(index=False))
orders = pd.read_csv(R / 'inventory_orders.csv')
assert (orders.purchase_spend <= orders.budget + 1e-5).all()
print('\nEvery review satisfies the purchasing budget.')
print('Optimisation solver status counts:')
print(orders.solver_status.dropna().value_counts().to_string())

               policy  budget_factor  lead_time  fill_rate  average_inventory_units  average_inventory_value  holding_cost  shortage_penalty  inventory_operating_cost  purchase_spend  stockout_item_days  budget_breaches
  Trailing mean cover            1.0          2   0.952660              3119.607143              3654.535143    306.980952          2467.080               2774.060952       51246.756                 285                0
       Forecast cover            1.0          2   0.953232              3159.654762              3758.757500    315.735630          2197.355               2513.090630       51388.464                 223                0
Scenario optimisation            1.0          2   0.972940              4181.690476              4813.717357    404.352258          1572.810               1977.162258       53128.206                 157                0

Every review satisfies the purchasing budget.
Optimisation solver status counts:
solver_status
0.0    108


## 7 Sensitivity and trade-offs

Nine combinations vary purchasing limits and replenishment lead time. Initial stock also changes with lead time; compare policies within each scenario, rather than interpreting cross-scenario differences as isolated causal effects.

![Inventory cost timeline](results/figures/inventory_timeline.png)

In [8]:
sensitivity = pd.read_csv(R / 'inventory_sensitivity.csv')
print(sensitivity.pivot(index=['lead_time','budget_factor'],columns='policy',values='inventory_operating_cost').to_string())

policy                   Forecast cover  Scenario optimisation  Trailing mean cover
lead_time budget_factor                                                            
0         0.8               9117.014262            8157.930718          9551.694206
          1.0               2460.635334            1839.111806          2920.751946
          1.2               2120.088730            1712.049856          2863.022356
2         0.8               8472.952394            7554.433002          8927.623676
          1.0               2513.090630            1977.162258          2774.060952
          1.2               2115.387050            1819.928548          2585.080198
4         0.8               8357.686172            7709.128010          8761.291206
          1.0               2537.229372            2007.601314          2717.090712
          1.2               2089.601874            1823.477594          2536.066840


## 8 Verification

Focused tests check future-sales isolation, forecast baselines, metric arithmetic, purchasing limits and a known small optimisation problem.

In [9]:
import subprocess, sys
result = subprocess.run([sys.executable, 'test_analysis.py'], capture_output=True, text=True, check=True)
print(result.stdout + result.stderr)

.....
----------------------------------------------------------------------
Ran 5 tests in 0.443s

OK



## Recommendation

Pilot the scenario-based ordering policy with actual retailer costs and inventory data. Report the additional working capital alongside simulated service gains. The model is a reproducible offline prototype; no live replenishment or realised company savings are claimed.